In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .appName("DeviceLevelAggregation") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
df = spark.read.csv("sensor_logs.csv", header=True, inferSchema=True)

In [4]:
print("Raw sensor readings:")
df.show()

Raw sensor readings:
+---------+---------------+-------------------+----+----------+
|device_id|    device_name|          timestamp|hour|energy_kwh|
+---------+---------------+-------------------+----+----------+
|        1|Air Conditioner|2026-01-01 14:00:00|  14|       2.5|
|        1|Air Conditioner|2026-01-01 20:00:00|  20|       2.3|
|        1|Air Conditioner|2026-01-02 15:00:00|  15|       2.7|
|        2|   Refrigerator|2026-01-01 00:00:00|   0|       1.1|
|        2|   Refrigerator|2026-01-01 12:00:00|  12|      1.05|
|        2|   Refrigerator|2026-01-02 01:00:00|   1|      1.15|
|        3|Washing Machine|2026-01-01 09:00:00|   9|       0.9|
|        3|Washing Machine|2026-01-02 10:00:00|  10|       1.0|
|        4|     Television|2026-01-01 19:00:00|  19|       0.3|
|        4|     Television|2026-01-02 21:00:00|  21|      0.25|
|        4|     Television|2026-01-03 22:00:00|  22|      0.35|
|        5|   Water Heater|2026-01-01 07:00:00|   7|       3.2|
|        5|   Water

In [5]:
df = df.withColumn(
    "period",
    F.when((F.col("hour") >= 8) & (F.col("hour") < 22), "peak").otherwise("off_peak")
)

In [6]:
peak_vs_offpeak = df.groupBy("device_id", "device_name", "period") \
    .agg(F.sum("energy_kwh").alias("total_kwh")) \
    .orderBy("device_id", "period")

In [7]:
print("Peak vs off-peak usage by device:")
peak_vs_offpeak.show()

Peak vs off-peak usage by device:
+---------+---------------+--------+---------+
|device_id|    device_name|  period|total_kwh|
+---------+---------------+--------+---------+
|        1|Air Conditioner|    peak|      7.5|
|        2|   Refrigerator|off_peak|     2.25|
|        2|   Refrigerator|    peak|     1.05|
|        3|Washing Machine|    peak|      1.9|
|        4|     Television|off_peak|     0.35|
|        4|     Television|    peak|     0.55|
|        5|   Water Heater|off_peak|      6.6|
|        5|   Water Heater|    peak|      6.0|
+---------+---------------+--------+---------+



In [8]:
top_devices = df.groupBy("device_id", "device_name") \
    .agg(F.sum("energy_kwh").alias("total_kwh")) \
    .orderBy(F.desc("total_kwh"))

In [9]:
print("Top energy-consuming devices:")
top_devices.show()

Top energy-consuming devices:
+---------+---------------+------------------+
|device_id|    device_name|         total_kwh|
+---------+---------------+------------------+
|        5|   Water Heater|12.600000000000001|
|        1|Air Conditioner|               7.5|
|        2|   Refrigerator|3.3000000000000003|
|        3|Washing Machine|               1.9|
|        4|     Television|               0.9|
+---------+---------------+------------------+



In [10]:
peak_vs_offpeak.toPandas().to_csv('peak_vs_offpeak.csv', index=False, header=True)
top_devices.toPandas().to_csv('top_devices.csv', index=False, header=True)

In [11]:
spark.stop()